# Demo 6: Jaynes-Cummings scan with photon-counting matrix criteria

This notebook mirrors `demo_3_jaynes_cummings.ipynb`, but uses the ideal photoelectric photon-counting matrix criterion `analytical_c` from Krishnaswamy et al. arXiv:2603.06237v1 as the displayed analytical comparison.

The first two examples show LP versus the counting-matrix criterion. The final selected-index example overlays all available boundaries: numerical LP, analytical K, analytical M, and the photon-counting matrix criterion.

Here "counting matrix" means the Sec. II ideal photoelectric matrix from Krishnaswamy et al. arXiv:2603.06237v1,

$$
C^{(I)} = \left[(k+l)! P_{k+l}\right]_{k,l\in I},
$$

not the multiplexed click-counting matrix.

Non-physical probability vectors, such as negative values or selected probabilities with sum larger than one, are skipped by the scan, stored as `NaN`, and drawn in white in the maps.


In [ ]:
# Make the project importable when running from the notebooks/ directory.
import os, sys
sys.path.insert(0, os.path.abspath('..'))

from src.config import apply_thread_limits
apply_thread_limits()

import numpy as np
import matplotlib.pyplot as plt


In [ ]:
from src.analytical.jaynes_cummings import Pjaycum
from src.comparison.scan_2d import analyze_2d, load_scan_result


## Helpers


In [ ]:
def make_jc_grid(tau, nu, pe, indx):
    """Build Pall[i_nu, i_tau, j] = P_{indx[j]}(nu[i_nu], tau[i_tau])."""
    xtau = tau[None, :]
    ynu = nu[:, None]
    return np.stack(
        [Pjaycum(n=int(n), nu=ynu, tau=xtau, pe=pe) for n in indx],
        axis=-1,
    )


def load_scan_archive(path):
    """Read a saved analyze_2d archive back into its components."""
    return load_scan_result(path)


def draw_boundary_curves(boundaries, *, color, label, mode='scatter', lw=2, s=4, min_points=5, max_jump=0.15, linestyle='-'):
    first = True
    for b in boundaries:
        if len(b) < min_points:
            continue

        if mode == 'scatter':
            plt.scatter(b[:, 0], b[:, 1], s=s, color=color, label=label if first else None)
        elif mode == 'line':
            plt.plot(b[:, 0], b[:, 1], lw=lw, color=color, linestyle=linestyle, label=label if first else None)
        elif mode == 'cut-line':
            jumps = np.sqrt(np.sum(np.diff(b, axis=0)**2, axis=1))
            cuts = np.where(jumps > max_jump)[0] + 1
            for piece in np.split(b, cuts):
                if len(piece) < min_points:
                    continue
                plt.plot(piece[:, 0], piece[:, 1], lw=lw, color=color, linestyle=linestyle, label=label if first else None)
                first = False
            continue
        else:
            raise ValueError("mode must be 'scatter', 'line', or 'cut-line'")

        first = False


def plot_cm_boundaries(data, title, mode='scatter'):
    plt.figure(figsize=(6, 5))
    draw_boundary_curves(data['numerical']['boundaries'], color='black', label='LP', mode=mode, lw=4, s=5)
    draw_boundary_curves(data['counting_matrix']['boundaries'], color='green', label='counting matrix', mode=mode, lw=3, s=4, linestyle='--')

    handles, labels = plt.gca().get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    if by_label:
        plt.legend(by_label.values(), by_label.keys())
    plt.xlim(0, float(np.max(data['x'])))
    plt.ylim(0, float(np.max(data['y'])))
    plt.xlabel(r'$\tau$')
    plt.ylabel(r'$\nu$')
    plt.title(title)
    plt.tight_layout()
    plt.show()


def plot_all_boundaries(data, title, mode='scatter'):
    plt.figure(figsize=(6, 5))
    draw_boundary_curves(data['numerical']['boundaries'], color='black', label='LP', mode=mode, lw=4, s=5)
    draw_boundary_curves(data['ana_k']['boundaries'], color='red', label='analytical K', mode=mode, lw=3, s=4)
    draw_boundary_curves(data['ana_m']['boundaries'], color='blue', label='analytical M', mode=mode, lw=3, s=4, linestyle=':')
    draw_boundary_curves(data['counting_matrix']['boundaries'], color='green', label='counting matrix', mode=mode, lw=3, s=4, linestyle='--')

    handles, labels = plt.gca().get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    if by_label:
        plt.legend(by_label.values(), by_label.keys())
    plt.xlim(0, float(np.max(data['x'])))
    plt.ylim(0, float(np.max(data['y'])))
    plt.xlabel(r'$\tau$')
    plt.ylabel(r'$\nu$')
    plt.title(title)
    plt.tight_layout()
    plt.show()


def classification_map(data, criterion='counting_matrix'):
    """0 classical, 1 analytical-only, 2 LP-only, 3 both, NaN non-physical."""
    num = data['numerical']['nonclass']
    ana = data['analytical']['nonclass'] if criterion == 'analytical' else data[criterion]['nonclass']
    valid = np.isfinite(num) & np.isfinite(ana)
    classes = np.full(num.shape, np.nan, dtype=float)
    classes[valid] = 0.0
    classes[valid & (ana > 0.5)] = 1.0
    classes[valid & (num > 0.5)] = 2.0
    classes[valid & (num > 0.5) & (ana > 0.5)] = 3.0
    return classes


def map_cmap():
    cmap = plt.get_cmap('viridis').copy()
    cmap.set_bad('white')
    return cmap


def plot_cm_map(data, title, boundary_mode='scatter'):
    classes = classification_map(data, criterion='counting_matrix')
    plt.figure(figsize=(6, 5))
    plt.pcolormesh(
        data['x'], data['y'],
        np.ma.masked_invalid(classes),
        shading='auto', cmap=map_cmap(),
    )
    plt.colorbar(label='0 classical / 1 CM-only / 2 LP-only / 3 both')
    draw_boundary_curves(data['numerical']['boundaries'], color='red', label='LP boundary', mode=boundary_mode, lw=2, s=3)
    plt.xlim(0, float(np.max(data['x'])))
    plt.ylim(0, float(np.max(data['y'])))
    plt.xlabel(r'$\tau$')
    plt.ylabel(r'$\nu$')
    plt.title(title)
    plt.tight_layout()
    plt.show()


def plot_all_analytical_map(data, title, boundary_mode='scatter'):
    classes = classification_map(data, criterion='analytical')
    plt.figure(figsize=(6, 5))
    plt.pcolormesh(
        data['x'], data['y'],
        np.ma.masked_invalid(classes),
        shading='auto', cmap=map_cmap(),
    )
    plt.colorbar(label='0 classical / 1 K-M-CM only / 2 LP-only / 3 both')
    draw_boundary_curves(data['numerical']['boundaries'], color='red', label='LP boundary', mode=boundary_mode, lw=2, s=3)
    draw_boundary_curves(data['analytical']['boundaries'], color='purple', label='K/M/CM boundary', mode=boundary_mode, lw=2, s=3, linestyle='--')
    plt.xlim(0, float(np.max(data['x'])))
    plt.ylim(0, float(np.max(data['y'])))
    plt.xlabel(r'$\tau$')
    plt.ylabel(r'$\nu$')
    plt.title(title)
    plt.tight_layout()
    plt.show()


def run_case(case, tau, nu):
    indx = np.asarray(case['indx'], dtype=int)
    Pall = make_jc_grid(tau=tau, nu=nu, pe=case['pe'], indx=indx)
    os.makedirs(os.path.dirname(os.path.abspath(case['fi'])), exist_ok=True)

    data = analyze_2d(
        Pall,
        x_values=tau,
        y_values=nu,
        indx=indx,
        fi=case['fi'],
        fixed_values=np.array([case['pe']]),
        use_ana_k=case.get('use_ana_k', False),
        use_ana_m=case.get('use_ana_m', False),
        use_cm=case.get('use_cm', True),
    )

    print(f"saved: {case['fi']}")
    print(f"grid: {Pall.shape[:2]} | indices: {indx.tolist()}")
    print(
        f"LP curves: {len(data['numerical']['boundaries'])} | "
        f"K curves: {len(data['ana_k']['boundaries'])} | "
        f"M curves: {len(data['ana_m']['boundaries'])} | "
        f"CM curves: {len(data['counting_matrix']['boundaries'])} | "
        f"CM fraction: {data['counting_matrix']['ncaf']:.4f}"
    )
    return data


## Grid

The default uses `stepx = stepy = 0.02`, giving about 500,000 grid points per scan. Increase these steps for an interactive first run.


In [ ]:
stepx = 0.02
stepy = 0.02
tau = np.arange(1e-3, 10.001, stepx)
nu = np.arange(1e-3, 20.001, stepy)

print('N_tau =', tau.shape[0], '| N_nu =', nu.shape[0], '| grid points =', tau.shape[0] * nu.shape[0])


## Example 1: $N_P=6$, $p_e=0.3$, all indices

This example displays the LP boundary against the photon-counting matrix boundary from Krishnaswamy et al. arXiv:2603.06237v1.


In [ ]:
case_1 = dict(
    pe=0.3,
    indx=np.arange(6),
    fi='../results/demo_6_jc_counting_matrix_NP6_pe03.npz',
    use_ana_m=False,
)

data = run_case(case_1, tau, nu)
plot_cm_boundaries(data, r'JC boundaries: counting matrix, $N_P=6$, $p_e=0.3$')
plot_cm_map(data, r'JC map: counting matrix, $N_P=6$, $p_e=0.3$')


## Example 2: $N_P=4,6,8,10,12$, $p_e=0.4$, all indices

Again, the displayed analytical comparison is the photon-counting matrix criterion from Krishnaswamy et al. arXiv:2603.06237v1.


In [ ]:
case_2 = dict(
    pe=0.40,
    indx=np.arange(4),
    fi='../results/demo_6_jc_counting_matrix_NP4_pe040.npz',
    use_ana_m=False,
)

data = run_case(case_2, tau, nu)
plot_cm_boundaries(data, r'JC boundaries: counting matrix, $N_P=4$, $p_e=0.40$')
plot_cm_map(data, r'JC map: counting matrix, $N_P=4$, $p_e=0.40$')


In [ ]:
case_2 = dict(
    pe=0.40,
    indx=np.arange(6),
    fi='../results/demo_6_jc_counting_matrix_NP6_pe040.npz',
    use_ana_m=False,
)

data = run_case(case_2, tau, nu)
plot_cm_boundaries(data, r'JC boundaries: counting matrix, $N_P=6$, $p_e=0.40$')
plot_cm_map(data, r'JC map: counting matrix, $N_P=6$, $p_e=0.40$')

In [ ]:
case_2 = dict(
    pe=0.40,
    indx=np.arange(8),
    fi='../results/demo_6_jc_counting_matrix_NP8_pe040.npz',
    use_ana_m=False,
)

data = run_case(case_2, tau, nu)
plot_cm_boundaries(data, r'JC boundaries: counting matrix, $N_P=8$, $p_e=0.40$')
plot_cm_map(data, r'JC map: counting matrix, $N_P=8$, $p_e=0.40$')

In [ ]:
case_2 = dict(
    pe=0.40,
    indx=np.arange(10),
    fi='../results/demo_6_jc_counting_matrix_NP10_pe040.npz',
    use_ana_m=False,
)

data = run_case(case_2, tau, nu)
plot_cm_boundaries(data, r'JC boundaries: counting matrix, $N_P=10$, $p_e=0.40$')
plot_cm_map(data, r'JC map: counting matrix, $N_P=10$, $p_e=0.40$')

In [ ]:
case_2 = dict(
    pe=0.40,
    indx=np.arange(12),
    fi='../results/demo_6_jc_counting_matrix_NP12_pe040.npz',
    use_ana_m=False,
)

data = run_case(case_2, tau, nu)
plot_cm_boundaries(data, r'JC boundaries: counting matrix, $N_P=12$, $p_e=0.40$')
plot_cm_map(data, r'JC map: counting matrix, $N_P=12$, $p_e=0.40$')

In [ ]:
# Inspect the named arrays stored in the latest NP=12 archive.
with np.load(case_2['fi'], allow_pickle=True) as archive:
    print(archive.files)
print(data.keys())


## Example 3: selected indices $[2,4,6,7,8]$, $p_e=0.4$

This final example compares all boundary families on the same Jaynes-Cummings grid: numerical LP, analytical K, analytical M, and the photon-counting matrix criterion. The map uses the combined analytical flag `K or M or counting matrix`.

For selected nonconsecutive indices, analytical M may need to generate majorization pairs on the fly, so this case can be slower than the first two examples.


In [ ]:
case_3 = dict(
    pe=0.4,
    indx=np.array([2, 4, 6, 7, 8]),
    fi='../results/demo_6_jc_counting_matrix_selected_2_4_6_7_8_pe04.npz',
    use_ana_k=True,
    use_ana_m=True,
    use_cm=True,
)

data = run_case(case_3, tau, nu)
plot_all_boundaries(data, r'JC boundaries: LP, K, M, and counting matrix, selected indices, $p_e=0.4$', mode='cut-line')
plot_all_analytical_map(data, r'JC map: K, M, and counting matrix, selected indices, $p_e=0.4$', boundary_mode='cut-line')


## Further examples

- [demo_1_lp_solver](demo_1_lp_solver.ipynb): Single-point LP witnesses, a Nelder–Mead cross-check, and a Jaynes–Cummings example.
- [demo_2_analytical_vs_numerical](demo_2_analytical_vs_numerical.ipynb): Probability-plane scans comparing LP with analytical K and M criteria.
- [demo_3_jaynes_cummings](demo_3_jaynes_cummings.ipynb): Jaynes–Cummings maps with K/M comparisons and plotting of multiple boundary branches.
- [demo_4_probabilistic_boson_addition](demo_4_probabilistic_boson_addition.ipynb): Probabilistic boson addition, reversed scan direction, tighter LP tolerances, probability-scale diagnostics, and CSV save/load; the two-boson example uses optional QuTiP.
- [demo_5_counting_matrix_vs_numerics](demo_5_counting_matrix_vs_numerics.ipynb): Probability-plane comparisons of LP, K, M, and counting-matrix criteria, including nonconsecutive indices and saved-scan plotting.
- [demo_7_vicinity_of_thermal_balance](demo_7_vicinity_of_thermal_balance.ipynb): Bisection near Jaynes–Cummings thermal balance, comparing K, counting matrices, and LP; cutoff comparisons and logarithmic-distance plots.
